# Lists and tuples: a beginner's guide 📚

**Wednesday · Library Loans project**

By the end, you should be able to:
1. Find items using **indexing** and **slicing**.
2. Explain the approximate cost of indexing, slicing, inserting and removing.
3. Choose between a **list** and a **tuple**.
4. Compare list and set membership using **100,000 book IDs**.

**How to use this notebook:** Run the cells from top to bottom. Click a cell and press **Shift + Enter**. Predict each result before running the cell. No extra Python packages are needed.

## 1. What is a list?

A list keeps items in order. It can be changed after it is created. Imagine a small shelf of books:

In [ ]:
books = ["Dune", "Matilda", "Hamlet", "Beloved"]
print(books)
print("Number of books:", len(books))

## 2. Indexing: pick one item

An **index** is an item's position. Python starts counting at **0**, not 1. A negative index counts from the end: `-1` is the last item.

| Index | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| Book | Dune | Matilda | Hamlet | Beloved |

**Ask the team:** What will `books[1]` and `books[-1]` return?

In [ ]:
print("First:", books[0])
print("Second:", books[1])
print("Last:", books[-1])

You can also use an index to change a list item. An index outside the list, such as `books[10]`, raises an `IndexError`.

**Cost:** Getting or replacing one item by index is **O(1)**: roughly constant time, even when the list grows. This does not mean it takes zero time.

In [ ]:
shelf = ["Dune", "Matilda", "Hamlet"]
shelf[1] = "Things Fall Apart"
print(shelf)

## 3. Slicing: pick a section

A slice has the form `items[start:stop]`. It includes the **start** position but stops **before** the stop position. Leaving out a position means “from the beginning” or “to the end”.

**Ask the team:** Which books will `books[1:3]` return?

In [ ]:
print("Positions 1 and 2:", books[1:3])
print("First two:", books[:2])
print("From position 2:", books[2:])
print("All books:", books[:])

**Cost:** A slice creates a new list containing `k` selected items. It takes **O(k)** time and uses **O(k)** extra space. For example, copying the whole list takes O(n), where `n` is the number of items.

A slice is a **shallow copy**: the new list is separate, but any nested objects inside it are shared.

In [ ]:
first_two = books[:2]
first_two[0] = "A different book"
print("Original:", books)
print("Slice:", first_two)

## 4. Adding and removing books

- `append(value)` adds to the end of a list.
- `insert(index, value)` puts an item at a position; later items may need to shift.
- `remove(value)` searches for and removes the **first matching value**. It raises `ValueError` if the value is missing.

**Ask the team:** Why might inserting at the beginning involve more work than adding at the end?

In [ ]:
shelf = ["Dune", "Matilda", "Hamlet"]
shelf.append("Beloved")
print("After append:", shelf)

shelf.insert(1, "Purple Hibiscus")
print("After insert:", shelf)

shelf.remove("Matilda")
print("After remove:", shelf)

### What does “cost” mean?

Big-O describes how the amount of work grows with input size. `n` is the number of items; `k` is the number of items in a slice. These are common Python list costs:

| Operation | Approximate cost | Why? |
|---|---|---|
| `books[i]` or `books[i] = value` | **O(1)** | Go directly to a position. |
| `books[start:stop]` | **O(k)** | Copy the selected `k` items. |
| `books.append(value)` | **O(1)** amortized | Usually add at the end; an occasional resize costs more. |
| `books.insert(i, value)` | **O(n)** worst case | Items after the position may shift. |
| `books.remove(value)` | **O(n)** worst case | Search for the value and shift later items. |
| `value in books` | **O(n)** worst case | May need to check every item. |

“Amortized” means averaged over many operations, not that every individual append takes the same time.

## 5. Tuples: a fixed collection

A tuple is ordered and can be indexed or sliced like a list. Unlike a list, you **cannot replace, add or remove its elements** after creating it. Use one for a small record that should not change, such as a book's ID and title.

**Ask the team:** Is a tuple a good choice for a shelf where books are added and removed every day?

In [ ]:
book_record = (101, "Dune")
print("Record:", book_record)
print("Book ID:", book_record[0])
print("Title:", book_record[1])
print("Slice:", book_record[:1])

Trying `book_record[0] = 102` would raise a `TypeError`; we do not run it here so the notebook keeps going. Tuple indexing is O(1), and slicing `k` items is O(k).

**Important:** A tuple cannot replace its elements, but a mutable object stored inside it (like a list) can still change. A one-item tuple needs a comma: `(101,)`.

**Rule of thumb:** Use a **list** for items that change; use a **tuple** for a fixed record.

## 6. Membership: is this book ID present?

The expression `item in collection` returns `True` or `False`. A list checks items in order. A **set** stores unique items and uses hashing for lookup; we will use it only for membership here.

A list membership check is **O(n)** in the worst case. A set membership check is **O(1) on average**, but can be **O(n) in the worst case**. Sets are useful for frequent membership checks, but they do not preserve list positions, and creating a set takes time and extra memory.

In [ ]:
small_list = [101, 102, 103]
small_set = {101, 102, 103}

print("102 in list?", 102 in small_list)
print("102 in set? ", 102 in small_set)
print("999 in set? ", 999 in small_set)

## 7. Benchmark: 100,000 book IDs ⏱️

We will compare the **same IDs** and the **same lookup** in a list and a set. We check an ID at the end of the list and an ID that is missing. Both make the list scan all 100,000 IDs.

We create both collections **before** timing. Each measurement repeats the same lookup 100 times. We repeat that measurement 3 times and report the fastest run, divided by 100: the approximate time **per lookup**. Timing will vary by computer. This measures lookup only—not the time or memory needed to create the set.

In [ ]:
import timeit

book_ids_list = list(range(100_000))
book_ids_set = set(book_ids_list)

last_id = 99_999
missing_id = 100_001
checks_per_run = 100
runs = 3

print("IDs in list:", len(book_ids_list))
print("IDs in set: ", len(book_ids_set))
print("Last ID found in both:", last_id in book_ids_list, last_id in book_ids_set)
print("Missing ID found in both:", missing_id in book_ids_list, missing_id in book_ids_set)

In [ ]:
def time_lookup(collection, book_id):
    """Return the best average seconds per membership check."""
    times = timeit.repeat(
        lambda: book_id in collection,
        number=checks_per_run,
        repeat=runs,
    )
    return min(times) / checks_per_run

for label, book_id in [("Last ID", last_id), ("Missing ID", missing_id)]:
    list_time = time_lookup(book_ids_list, book_id)
    set_time = time_lookup(book_ids_set, book_id)

    print(label)
    print(f"  List: {list_time * 1_000_000:.2f} microseconds per check")
    print(f"  Set:  {set_time * 1_000_000:.2f} microseconds per check")

**Discuss:** Which was faster on your computer? Why? Would the list result change if we searched for the first ID instead? (Try changing `last_id` to `0` and rerun both benchmark cells.) A set is often worthwhile for many lookups, but its setup cost may not be worthwhile for a single lookup.

## 8. Practice together ✍️

Use the list below. First predict, then add your own code in the next cell:

1. Print the first and last title.
2. Print a slice containing `"Matilda"` and `"Hamlet"`.
3. Add `"Beloved"` at the end.
4. Remove `"Dune"` by value.
5. Create a tuple containing book ID `101` and title `"Dune"`.

Run the first cell again whenever you want to reset the practice list.

In [ ]:
practice_books = ["Dune", "Matilda", "Hamlet"]

In [ ]:
# Write your answers here.
# Example: print(practice_books[0])

### Suggested solution

Try the exercise first. This example starts with a fresh list so you can run it independently.

In [ ]:
practice_books = ["Dune", "Matilda", "Hamlet"]
print(practice_books[0], practice_books[-1])
print(practice_books[1:3])
practice_books.append("Beloved")
practice_books.remove("Dune")
record = (101, "Dune")
print("Updated list:", practice_books)
print("Tuple:", record)

## 9. Quick knowledge-check function ✅

For the following questions, imagine `titles = ["Dune", "Matilda", "Hamlet"]`.

1. What is `titles[0]`? Give a string.
2. What is `titles[1:3]`? Give a list of strings.
3. Which collection is usually faster for repeated membership checks: `"list"` or `"set"`?
4. Which is a better fit for a fixed book record: `"list"` or `"tuple"`?
5. What is the **worst-case** cost of removing a value from a list? Give `"O(n)"` or `"O(1)"`.

Run the function cell. Then replace the example answers in the next cell with your own and run it. The checker prints feedback; it does not change the book list.

In [ ]:
def check_understanding(first_title, selected_titles, membership_choice,
                        record_choice, remove_cost):
    """Check five answers and print a simple score."""
    answers = [
        first_title == "Dune",
        selected_titles == ["Matilda", "Hamlet"],
        membership_choice == "set",
        record_choice == "tuple",
        remove_cost == "O(n)",
    ]
    for number, correct in enumerate(answers, start=1):
        if correct:
            print(f"Question {number}: Correct!")
        else:
            print(f"Question {number}: Try again.")
    print(f"Score: {sum(answers)} out of {len(answers)}")

In [ ]:
# Replace these example answers with your own before running.
check_understanding(
    first_title="your answer",
    selected_titles=[],
    membership_choice="your answer",
    record_choice="your answer",
    remove_cost="your answer",
)

## 10. Presenter question round: 20 quick questions

Ask the team to answer aloud or in chat before opening the answer key.

1. What is a Python list?
2. What index is used for the first item?
3. What does index `-1` mean?
4. What happens if you use an index that is out of range?
5. What is the typical Big-O cost of reading a list item by index?
6. In `items[1:3]`, is the item at index 3 included?
7. How many items does `items[1:3]` select if the list has at least 3 items?
8. What does `items[:2]` select?
9. Does slicing a list create a separate list?
10. What are the time and extra-space costs of slicing `k` items?
11. What does `append` do, and what is its amortized cost?
12. What does `insert` do, and why can it cost O(n)?
13. Does `remove` take an index or a value?
14. What does `remove` do when there are repeated matching values?
15. What happens if `remove` cannot find the value?
16. What is the worst-case cost of removing a value from a list?
17. Can you replace an element of a tuple after creation?
18. When might a tuple be a better choice than a list?
19. What are the worst-case costs of list and set membership checks?
20. What does our benchmark exclude by creating the set before timing?

<details>
<summary><strong>Open the answer key</strong></summary>

1. An ordered, changeable collection of items.
2. `0`.
3. The last item.
4. Python raises `IndexError`.
5. O(1).
6. No; the stop is excluded.
7. Two: positions 1 and 2.
8. Items from the start up to, but not including, index 2.
9. Yes; it is a shallow copy.
10. O(k) time and O(k) extra space.
11. Adds at the end; O(1) amortized.
12. Adds at a position; later items may need to shift.
13. A value.
14. It removes the first match.
15. Python raises `ValueError`.
16. O(n).
17. No.
18. For a fixed record, such as a book ID and title.
19. List: O(n); set: O(n) worst case, though O(1) on average.
20. The time and memory cost of constructing the set.

</details>

## Takeaway

- **Index:** one item, O(1). **Slice:** copy `k` items, O(k).
- **List:** changeable. Inserting/removing can be O(n).
- **Tuple:** fixed elements; useful for a record that should not change.
- **Set:** useful for repeated membership checks; O(1) on average, with setup and memory costs.